# S2c baselines on Colab (E13 pre-registration v1, section 6; B7)

Runs the tuned MaxPressure grid, the Webster plans, the queue log of the fixed and actuated plans and the B7 baseline episodes with `s2c.py` from `s2c_kit.zip` (SUMO 1.26.0 + libsumo). Every episode is one result file written atomically to `OUT` on Google Drive; a cell that stops (disconnect, timeout) is simply run again and continues with the missing episodes.

Order: 1 Drive, 2 settings, 3 install, 4 unzip and checksum, 5 smoke (must say `"passed": true`), 6 runs (tune first: eval and b7 need `tune/tuned.json`), 7 status. The locked test (cell 8) runs **once**, only when the lead asks, after the E13 selection is logged.

Upload `s2c_kit.zip` and `s2c_kit.zip.sha256` to `MyDrive/S2c/` first.

In [ ]:
# 1. Google Drive (click the consent dialog)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Settings
KIT_ZIP = '/content/drive/MyDrive/S2c/s2c_kit.zip'
OUT = '/content/drive/MyDrive/S2c/out'      # results (resumable); the smoke test writes to local disk
JOBS = 8                                    # parallel SUMO processes (Colab Pro High-RAM: 8 vCPU)
import os; os.environ.update(KIT_ZIP=KIT_ZIP, OUT=OUT, JOBS=str(JOBS))

In [ ]:
# 3. Install SUMO 1.26.0 with libsumo (the versions of every E13 run)
!pip -q install eclipse-sumo==1.26.0 libsumo==1.26.0
!python -c "import libsumo, sumo, platform; print('libsumo OK, SUMO_HOME', sumo.SUMO_HOME, platform.python_version())"

In [ ]:
# 4. Unzip the kit and check every file against KIT_MANIFEST.sha256
import hashlib, pathlib
digest = hashlib.sha256(open(KIT_ZIP, 'rb').read()).hexdigest()
exp = pathlib.Path(KIT_ZIP + '.sha256')
print('zip sha256', digest, '| expected', exp.read_text().split()[0] if exp.exists() else '(no .sha256 file uploaded)',
      '| OK' if exp.exists() and exp.read_text().split()[0] == digest else '')
!rm -rf /content/s2c_kit && unzip -q -o "$KIT_ZIP" -d /content/
%cd /content/s2c_kit
!python s2c.py verify-kit

In [ ]:
# 5. Smoke test (about 1 min): fixed and actuated identical to E13's baselines, MaxPressure and Webster signal rules,
#    queue log of 360 samples. Must end with "passed": true. Then the Webster plans recomputed and compared.
!python s2c.py smoke --out /content/s2c_smoke --jobs $JOBS | tail -40
!python s2c.py plan --out /content/s2c_smoke

In [ ]:
# 6a. Tuning: 3 loads x 20 training files x 5 seeds x (12 MaxPressure grid points + fixed plan) = 3900 episodes
!python s2c.py tune --out "$OUT" --jobs $JOBS

In [ ]:
# 6b. Tuned MaxPressure and Webster on validation and test A (880 episodes)
!python s2c.py eval --set validation --out "$OUT" --jobs $JOBS
!python s2c.py eval --set testA --out "$OUT" --jobs $JOBS

In [ ]:
# 6c. Queue log of the fixed and actuated plans on validation and test A (880 episodes)
!python s2c.py queue --set validation --out "$OUT" --jobs $JOBS
!python s2c.py queue --set testA --out "$OUT" --jobs $JOBS

In [ ]:
# 6d. B7: fixed, actuated, tuned MaxPressure on every B7 file in the kit (k = 1)
!python s2c.py b7 --out "$OUT" --jobs $JOBS

In [ ]:
# 7. Status (episodes done per stage, tuned configuration, seconds per episode); tables in OUT/results/
!python s2c.py status --out "$OUT"

In [ ]:
# 8. LOCKED TEST: once, only when the lead asks, after the E13 selection is logged. Remove the '#' to run.
# !python s2c.py eval  --set locked --locked-ok --out "$OUT" --jobs $JOBS
# !python s2c.py queue --set locked --locked-ok --out "$OUT" --jobs $JOBS